Name: Ricardo Flores <br>
Model: Forma-1 <br>
Dataset: RICO Mobile UI Screenshots

# Forma-1 — Mobile UI Diffusion Model

### 1. Environment setup

In [ ]:
# confirm GPU assignment — must show A100 or H100 before proceeding
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# clone the repo into the colab runtime
!git clone https://github.com/yourusername/forma-1.git
%cd forma-1

# install dependencies from requirements.txt
!pip install -r requirements.txt --quiet

# add project root to python path so local modules can be imported
import sys
sys.path.append('/content/forma-1')

# mount google drive — dataset and checkpoints live here across sessions
from google.colab import drive
drive.mount('/content/drive')

### 2. Imports

In [ ]:
# standard library
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# tensorflow + mixed precision
import tensorflow as tf
from tensorflow.keras import mixed_precision

# enable float16 on A100/H100 tensor cores — ~2x throughput, no quality loss
mixed_precision.set_global_policy('mixed_float16')

# forma-1 modules
from data.preprocess      import get_dataset
from model.noise_schedule import get_noise_schedule
from model.unet           import get_unet
from training.train       import run_training
from sampling.sample      import (sample_ddpm, sample_ddim,
                                   sample_with_steps, plot_grid,
                                   plot_denoising_steps)

### 3. Configure paths and hyperparameters

In [ ]:
# paths — update these to match your Drive folder structure
DRIVE_DATA_PATH = '/content/drive/MyDrive/DiffuseUI/Dataset/RICO/unique_uis/combined'
CHECKPOINT_DIR  = '/content/drive/MyDrive/DiffuseUI/checkpoints'
EXPORT_DIR      = '/content/drive/MyDrive/DiffuseUI/model'

# image resolution — 128x128 shows recognizable UI structure (nav bars, cards, buttons)
IMAGE_SIZE = 128

# diffusion schedule
T          = 1000
BETA_START = 1e-4
BETA_END   = 0.02

# time embedding dimension — sinusoidal encoding of the current timestep
TIME_DIM = 256

# training settings
BATCH_SIZE    = 32
EPOCHS        = 200
LEARNING_RATE = 1e-4

# create output directories if they don't already exist
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR,     exist_ok=True)

print('Image size:   ', IMAGE_SIZE)
print('Batch size:   ', BATCH_SIZE)
print('Epochs:       ', EPOCHS)
print('Timesteps:    ', T)
print('Precision:    ', mixed_precision.global_policy().name)

### 4. Load and preview the dataset

In [ ]:
# build the tf.data pipeline from the RICO screenshots on Drive
dataset, num_images = get_dataset(DRIVE_DATA_PATH, IMAGE_SIZE, BATCH_SIZE)
print(f'Batches per epoch: {len(dataset)}')

# confirm pixel range after preprocessing — should be [-1, 1]
sample_batch = next(iter(dataset))
print('Pixel range:', sample_batch.numpy().min(), 'to', sample_batch.numpy().max())

# preview 9 preprocessed samples
plt.figure(figsize=(9, 9))
for i in range(9):
    img = (sample_batch[i].numpy() + 1.0) / 2.0   # convert back to [0, 1] for display
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.axis('off')

plt.suptitle('Sample RICO UI Screenshots (preprocessed)')
plt.tight_layout()
plt.show()

### 5. Build the noise schedule

In [ ]:
# compute beta, alpha, and alpha_bar for the full 1000-step linear schedule
beta, alpha, alpha_bar = get_noise_schedule(T, BETA_START, BETA_END)

### 6. Build the U-Net

In [ ]:
# instantiate the U-Net denoiser and Adam optimizer
model     = get_unet(image_size=IMAGE_SIZE, time_dim=TIME_DIM)
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.summary()

### 7. Resume from checkpoint (if available)

In [ ]:
weights_path = os.path.join(CHECKPOINT_DIR, 'forma1_weights.weights.h5')
history_path = os.path.join(CHECKPOINT_DIR, 'forma1_history.npy')

start_epoch = 0
all_losses  = []

# check for an existing checkpoint and resume if found
if os.path.exists(weights_path):
    model.load_weights(weights_path)
    if os.path.exists(history_path):
        all_losses  = list(np.load(history_path))
        start_epoch = len(all_losses)
    print(f'Checkpoint loaded — resuming from epoch {start_epoch}')
else:
    print('No checkpoint found — starting fresh')

### 8. Train the model

In [ ]:
# run the full training loop — checkpoints saved to Drive every 10 epochs
all_losses = run_training(
    model        = model,
    optimizer    = optimizer,
    dataset      = dataset,
    alpha_bar    = alpha_bar,
    T            = T,
    time_dim     = TIME_DIM,
    epochs       = EPOCHS,
    checkpoint_dir = CHECKPOINT_DIR,
    start_epoch  = start_epoch,
    all_losses   = all_losses
)

### 9. Plot the training loss

In [ ]:
plt.figure()
plt.plot(all_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Forma-1 Training Loss')
plt.tight_layout()
plt.show()

### 10. Generate UI screenshots — DDPM (1000 steps)

In [ ]:
# full reverse diffusion — 1000 steps, follows the original DDPM paper exactly
generated = sample_ddpm(
    model      = model,
    num_samples = 36,
    image_size = IMAGE_SIZE,
    T          = T,
    beta       = beta,
    alpha      = alpha,
    alpha_bar  = alpha_bar,
    time_dim   = TIME_DIM
).numpy()

plot_grid(generated, 'Forma-1 Generated UI Screenshots (DDPM, 1000 steps)')

### 11. Generate UI screenshots — DDIM (50 steps)

In [ ]:
# fast deterministic sampler — 50 steps instead of 1000, ~20x faster generation
# uses the same trained model, no retraining needed
generated_fast = sample_ddim(
    model       = model,
    num_samples = 36,
    image_size  = IMAGE_SIZE,
    T           = T,
    alpha_bar   = alpha_bar,
    time_dim    = TIME_DIM,
    num_steps   = 50
).numpy()

plot_grid(generated_fast, 'Forma-1 Generated UI Screenshots (DDIM, 50 steps)')

### 12. Visualize the denoising process

In [ ]:
# run the full reverse process on a single image and capture 10 snapshots
# shows the progression from pure gaussian noise to a generated UI screenshot
snapshots = sample_with_steps(
    model         = model,
    image_size    = IMAGE_SIZE,
    T             = T,
    beta          = beta,
    alpha         = alpha,
    alpha_bar     = alpha_bar,
    time_dim      = TIME_DIM,
    num_snapshots = 10
)

plot_denoising_steps(snapshots)

### 13. Export the model

In [ ]:
# save the full model in keras format
model.save(os.path.join(EXPORT_DIR, 'forma1_final.keras'))

# save a config file with all architecture and training details
config = {
    'model_name':          'Forma-1',
    'image_size':          IMAGE_SIZE,
    'timesteps':           T,
    'beta_start':          BETA_START,
    'beta_end':            BETA_END,
    'beta_schedule':       'linear',
    'time_embedding_dim':  TIME_DIM,
    'encoder_channels':    [64, 128, 256, 512],
    'attention_at':        ['bottleneck_8x8', 'decoder_16x16'],
    'batch_size':          BATCH_SIZE,
    'epochs':              EPOCHS,
    'learning_rate':       LEARNING_RATE,
    'mixed_precision':     'float16',
    'dataset':             'RICO Mobile UI Screenshots',
    'num_training_images': num_images,
    'framework':           'TensorFlow / Keras',
    'license':             'creativeml-openrail-m'
}

config_path = os.path.join(EXPORT_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print('Model saved to:  ', EXPORT_DIR)
print('Config saved to: ', config_path)